# Lab: Single Agent Baseline with Azure OpenAI and LangChain
## Lab Overview: What this lab is and why it matters
This lab establishes a **single-agent baseline** using Azure OpenAI and LangChain. A baseline agent is the foundational unit from which every more complex architecture grows. Before you can reason about multi-agent systems, MCP orchestration, or GitHub Copilot Agent Mode, you need a precise understanding of what a single agent does, how it manages state across turns, how it invokes tools, and where its limits lie.

The lab is grounded in five theoretical areas that map directly to real enterprise patterns: Assistants and Tool Calling, Threading, Tool Invocation Patterns, Multi-Agent Architectures, and MCP with GitHub Copilot Agent Mode. Each section introduces the theory before the code so you understand not just what to type but why each design decision was made.

All examples use finance and healthcare domains because these industries have the most demanding requirements around auditability, determinism, and compliance — exactly the conditions that stress-test agent design.
## What You Will Learn: Skills and understanding this lab builds
By completing this lab you will be able to:

1. Explain the Assistants abstraction and how tool calling differs from plain completion.
2. Understand threading as a state management mechanism and implement thread-scoped memory.
3. Identify and implement the three primary tool invocation patterns: sequential, parallel, and conditional.
4. Understand where a single agent ends and a multi-agent architecture begins, and articulate the trade-offs.
5. Explain MCP (Model Context Protocol) as a standardisation layer for tool connectivity.
6. Understand GitHub Copilot Agent Mode and how it relates to the patterns built in this lab.
## Prerequisites: What you need before starting
- An Azure OpenAI resource with a deployed GPT-4o model.
- The deployment name, endpoint URL, and API key for that resource.
- Python 3.10 or later. Python 3.12 is recommended.
- The previous lab (LangChain and Azure OpenAI agent foundations) completed, or equivalent familiarity with `AzureChatOpenAI`, `create_react_agent`, and `MemorySaver`.
- A Jupyter-compatible environment.

## Section 1: Theory — Assistants and Tool Calling
### 1.1 What the Assistants Abstraction Provides
The term **Assistant** in the context of LLM APIs refers to a model instance that has been given a persistent identity through a system prompt, a set of available tools, and optionally a memory store. The distinction from a raw completion endpoint is significant:

A raw completion endpoint is stateless. You send a prompt, you receive a completion. Nothing persists. Every call is independent.

An Assistant is stateful by design. It has an identity (system prompt), capabilities (tools), and a conversation history (thread). When you interact with an Assistant, the model reasons within the context of all three.

In enterprise terms, think of the difference between calling a financial data vendor's API directly (stateless, one data point at a time) versus engaging a dedicated analyst who knows your portfolio, your risk tolerance, and your prior questions. The Assistant abstraction is the latter.
### 1.2 How Tool Calling Works at the Protocol Level
When a model is given tools, each tool is serialised as a JSON Schema object describing its name, description, and parameter types. This schema is sent to the model alongside the user message. The model does not execute code. It returns a structured JSON object indicating which tool it wants to call and with what arguments.

The execution layer (LangChain's `ToolNode` in this lab) receives that JSON, calls the actual Python function, captures the return value, and feeds it back to the model as a `ToolMessage`. The model then continues reasoning.

This separation is architecturally important. The model is a reasoning engine, not an execution engine. The tool call is a request, not a command. Your code decides whether to honour it, which is how human-in-the-loop patterns are implemented.
### 1.3 Tool Calling vs. Function Calling vs. Plugins
These three terms describe the same underlying mechanism at different points in its evolution:

**Function calling** was the original OpenAI term (2023). The model emits a `function_call` field in its response.

**Tool calling** is the current standardised term. The model emits a `tool_calls` array, supporting multiple parallel calls in a single response.

**Plugins** (OpenAI ChatGPT Plugins, now deprecated) were a hosted variant where tools were served over HTTP with an OpenAPI spec. MCP (Section 5) is the successor to this pattern.

LangChain's `@tool` decorator and the `AzureChatOpenAI.bind_tools()` method work with the current tool calling standard.

## Section 2: Theory — Threading
### 2.1 What a Thread Is
A **thread** is the unit of conversation state. Every message the user sends, every tool call the agent makes, every tool result returned, and every model response — all of these are `Message` objects appended to a thread. The thread is the context window with a persistent identity.

In LangGraph, a thread is identified by a `thread_id` string stored in the `configurable` key of the invocation config. The checkpointer (memory store) uses this key to load the correct state before each invocation and save the updated state after.

In a healthcare setting, a thread maps naturally to a clinical encounter or a patient session. A clinician querying drug interactions, then following up about dosing, then asking about monitoring parameters is a single thread. Each follow-up benefits from the prior context without the clinician re-stating the patient situation.
### 2.2 Thread Lifecycle
A thread begins with the first message that carries its `thread_id`. It persists as long as the checkpointer retains its state. It ends when the state is deleted or expires.

Three lifecycle operations matter in production:

**Create:** First invocation with a new `thread_id`. No prior state exists.

**Continue:** Subsequent invocations with the same `thread_id`. Full history is loaded and the new message is appended.

**Fork:** Branching from a checkpoint, replaying from a specific point in history. Used for what-if analysis ("what if the agent had taken a different action at step 3?").
### 2.3 Threading and Compliance
In financial services, thread state is an audit record. Under MiFID II and similar frameworks, a firm must be able to reproduce the reasoning chain behind a recommendation. A thread with its full message history, including tool calls and observations, satisfies this requirement if stored correctly. `MemorySaver` is in-memory only and does not persist across process restarts. Production deployments use PostgreSQL or a document store checkpointer.

## Section 3: Theory — Tool Invocation Patterns
### 3.1 Sequential Invocation
The model calls tools one at a time, using the result of each to inform whether and how to call the next. This is the default ReAct pattern. It is the safest pattern because each step is observable before the next is taken, but it is the slowest because steps cannot overlap.

Example: A risk analyst asks the agent to assess a bond position. The agent fetches the current yield, then uses that yield to calculate duration, then fetches the issuer credit rating. Each step depends on the previous.
### 3.2 Parallel Invocation
GPT-4o can return multiple tool calls in a single response when it determines they are independent. LangChain's `ToolNode` executes them concurrently and returns all results before the model's next turn. This reduces latency significantly for multi-data-point queries.

Example: Fetch the prices of AAPL, MSFT, and JPM simultaneously. None depends on the others. A single model turn produces three tool calls; a single `ToolNode` execution produces three results.
### 3.3 Conditional Invocation
The model reasons about whether a tool call is necessary at all before invoking it. This is not a separate API feature but an emergent behaviour from the system prompt and the model's reasoning. A well-designed agent does not call tools it does not need. A poorly designed one calls every tool on every turn.

Example: A user asks "what is 15% of $200,000?" — the agent should answer directly without invoking any financial data tool, because no external data is needed. This saves tokens, latency, and cost.
### 3.4 Recursive and Self-Directed Invocation
An agent can call a tool whose output triggers a further tool call in the same reasoning chain. This is normal in complex queries. The danger is unbounded recursion. LangGraph's `recursion_limit` parameter (default 25) is the circuit breaker. In production, set this explicitly based on the maximum expected depth of your use case.

## Section 4: Theory — Multi-Agent Architectures
### 4.1 Why a Single Agent Is Not Always Enough
A single agent with many tools works well up to a point. Beyond roughly 10 to 15 tools, tool selection quality degrades because the model's attention is spread across too many options. The system prompt grows unwieldy. Error recovery becomes complex because a single point of failure affects the entire task.

Multi-agent architectures solve this through specialisation and delegation. Each agent has a focused set of tools and a narrow system prompt. A supervisor or orchestrator agent receives the user's request, decomposes it, and routes sub-tasks to specialist agents.
### 4.2 Common Multi-Agent Topologies
**Supervisor pattern:** One orchestrator receives all user input. It routes to specialists (a finance agent, a healthcare agent, a document retrieval agent) and synthesises their outputs. This lab's single-agent baseline is the building block for each specialist in this pattern.

**Peer pattern:** Multiple agents with equal authority collaborate on a shared message channel. Each contributes when its specialty is relevant. This is harder to orchestrate but more flexible.

**Pipeline pattern:** Agents form a chain where each processes the output of the previous. Common in document processing: an extraction agent produces structured data that a validation agent checks, whose output a reporting agent formats.
### 4.3 The Single-Agent Baseline as a Design Unit
Every agent in a multi-agent system is a single-agent baseline with a scoped system prompt and a curated tool set. Building a robust single-agent baseline — one with well-described tools, correct memory configuration, proper error handling, and token monitoring — is the prerequisite skill. You cannot build a reliable multi-agent system from unreliable single agents.

This is why this lab focuses entirely on getting the single-agent baseline right before introducing orchestration.

## Section 5: Theory — MCP and GitHub Copilot Agent Mode
### 5.1 What MCP Is
**Model Context Protocol (MCP)** is an open standard (Anthropic, 2024) that defines a uniform interface for connecting LLMs to external tools, data sources, and services. Before MCP, every LLM framework defined its own tool interface. LangChain tools could not be directly used in a Semantic Kernel agent without adaptation, and vice versa.

MCP addresses this by defining a standard JSON-RPC protocol for tool discovery, invocation, and result return. An MCP server exposes tools. An MCP client (the LLM agent) discovers and calls them. Any agent that speaks MCP can use any MCP server regardless of the underlying framework.

In enterprise terms, MCP is the USB standard for AI tool connectivity. Once your data or service exposes an MCP server, it is accessible to any compliant agent without custom integration work.
### 5.2 MCP Architecture
An MCP deployment has three components:

**MCP Host:** The application running the LLM agent (for example, an Azure-hosted LangChain application, or VS Code with Copilot).

**MCP Client:** The protocol client inside the host that manages the connection to MCP servers.

**MCP Server:** A process that exposes tools, resources (files, database rows, API results), and prompts over the MCP protocol. Servers can be local (a Python subprocess) or remote (an HTTP/SSE endpoint).

The tools you build in this lab follow the same design principles as MCP tool implementations. The `@tool` decorator produces a schema that is structurally identical to an MCP tool definition.
### 5.3 GitHub Copilot Agent Mode
GitHub Copilot Agent Mode (released 2025) extends Copilot from a code completion assistant to a full agent that can perform multi-step tasks: reading files, running terminal commands, editing code, running tests, and iterating on failures. It operates within VS Code and uses MCP to connect to external tools.

The architecture of a Copilot Agent Mode session is identical to the single-agent baseline in this lab:

- There is a system prompt (Copilot's instructions plus the workspace context).
- There is a thread (the current agent session).
- There are tools (file read, file write, terminal, MCP servers).
- There is a ReAct loop driving execution.

Understanding the patterns in this lab — threading, tool invocation, error handling, token management — directly transfers to designing and debugging Copilot Agent Mode workflows. When a Copilot agent gets stuck in a loop or calls the wrong tool, the diagnosis is the same as for a LangChain agent: inspect the thread, check the tool descriptions, add a recursion limit.

## Step 1: Install and Verify Packages
Install all LangChain ecosystem packages in a single command. Do not pin individual versions separately. The ecosystem's sub-packages (`langchain-core`, `langgraph-prebuilt`, etc.) must resolve to a mutually compatible set. Pinning them independently is the most common cause of `ImportError` failures in this ecosystem.

Restart the kernel after this cell completes.

In [1]:
%pip install \
    langchain \
    langchain-openai \
    langchain-community \
    langgraph \
    openai \
    python-dotenv \
    tiktoken \
    --upgrade --quiet

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 112.4/112.4 kB 7.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 87.7/87.7 kB 8.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.5/2.5 MB 51.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 167.5/167.5 kB 14.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.0/1.0 MB 50.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 503.0/503.0 kB 37.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 64.7/64.7 kB 5.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 51.0/51.0 kB 4.7 MB/s eta 0:00:00
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
google-colab 1.0.0 requires requests==2.32.4, but you have requests 2.32.5 which is incompatible.


> Restart the kernel before running any cell below this line.

In [2]:
import importlib, sys

print(f"Python: {sys.version}")
print()

for pkg in ["langchain_core", "langchain", "langchain_openai", "langgraph", "openai"]:
    try:
        m = importlib.import_module(pkg)
        print(f"{pkg:<30} {getattr(m, '__version__', 'unknown')}")
    except ImportError:
        print(f"{pkg:<30} NOT INSTALLED")

print()
checks = [
    ("langgraph.prebuilt",          "create_react_agent"),
    ("langgraph.checkpoint.memory", "MemorySaver"),
    ("langchain_openai",             "AzureChatOpenAI"),
    ("langchain_core.tools",         "tool"),
    ("langchain_core.messages",      "HumanMessage"),
]
all_ok = True
for module_path, symbol in checks:
    try:
        getattr(importlib.import_module(module_path), symbol)
        print(f"  OK   from {module_path} import {symbol}")
    except (ImportError, AttributeError) as e:
        print(f"  FAIL from {module_path} import {symbol} -> {e}")
        all_ok = False

print()
print("All imports healthy." if all_ok else "Fix: re-run the install cell, restart kernel, retry.")

Python: 3.12.12 (main, Oct 10 2025, 08:52:57) [GCC 11.4.0]

langchain_core                 1.2.18
langchain                      1.2.12
langchain_openai               unknown
langgraph                      unknown
openai                         2.26.0

  OK   from langgraph.prebuilt import create_react_agent
  OK   from langgraph.checkpoint.memory import MemorySaver
  OK   from langchain_openai import AzureChatOpenAI
  OK   from langchain_core.tools import tool
  OK   from langchain_core.messages import HumanMessage

All imports healthy.


## Step 2: Configure Credentials
Create a `.env` file in the same directory as this notebook:
```
AZURE_OPENAI_API_KEY=your_api_key
AZURE_OPENAI_ENDPOINT=https://your-resource.openai.azure.com/
AZURE_OPENAI_DEPLOYMENT_NAME=gpt-4o
AZURE_OPENAI_API_VERSION=2025-01-01-preview
```

In [3]:
from pathlib import Path

env_content = """
AZURE_OPENAI_ENDPOINT=https://<TBD>.cognitiveservices.azure.com
AZURE_OPENAI_API_KEY=
AZURE_OPENAI_DEPLOYMENT_NAME=gpt-4o
AZURE_OPENAI_API_VERSION=2024-12-01-preview
"""

# Create .env file in current directory
env_path = Path(".env")
env_path.write_text(env_content.strip())

print(".env file created successfully.")
print(env_path.resolve())

.env file created successfully.
/content/.env


In [4]:
import os
from dotenv import load_dotenv

load_dotenv(override=True)

required = [
    "AZURE_OPENAI_API_KEY",
    "AZURE_OPENAI_ENDPOINT",
    "AZURE_OPENAI_DEPLOYMENT_NAME",
    "AZURE_OPENAI_API_VERSION",
]
missing = [v for v in required if not os.getenv(v)]
if missing:
    raise EnvironmentError(f"Missing variables: {missing}. Populate your .env file.")

print("Credentials loaded.")
print(f"  Endpoint:   {os.getenv('AZURE_OPENAI_ENDPOINT')}")
print(f"  Deployment: {os.getenv('AZURE_OPENAI_DEPLOYMENT_NAME')}")
print(f"  Version:    {os.getenv('AZURE_OPENAI_API_VERSION')}")

Credentials loaded.
  Endpoint:   https://<TBD>.cognitiveservices.azure.com
  Deployment: gpt-4o
  Version:    2024-12-01-preview


## Step 3: Instantiate the Model
`AzureChatOpenAI` implements `BaseChatModel`, the same interface used by every other LangChain-supported provider. All agent code written against this interface is portable to other providers by swapping this single object. `temperature=0` is mandatory for agents — non-zero temperature introduces randomness into tool selection decisions, which makes agent behaviour non-deterministic and harder to debug.

In [5]:
from langchain_openai import AzureChatOpenAI
from langchain_core.messages import HumanMessage

llm = AzureChatOpenAI(
    azure_deployment=os.getenv("AZURE_OPENAI_DEPLOYMENT_NAME"),
    azure_endpoint=os.getenv("AZURE_OPENAI_ENDPOINT"),
    api_key=os.getenv("AZURE_OPENAI_API_KEY"),
    api_version=os.getenv("AZURE_OPENAI_API_VERSION"),
    temperature=0,
    max_tokens=4096,
)

# Connectivity check
resp = llm.invoke([HumanMessage(
    content="In one sentence: what does Tier 1 capital measure in Basel III?"
)])
print("Connectivity check passed.")
print(resp.content)

Connectivity check passed.
Tier 1 capital in Basel III measures a bank's core financial strength, primarily consisting of common equity and disclosed reserves, to absorb losses and ensure stability during financial stress.


## Step 4: Define the Tool Set — Finance and Healthcare Domains
Each tool is a Python function decorated with `@tool`. The docstring is parsed into the JSON Schema that is sent to the model. Three elements of a good tool docstring: what it does, when to use it (and when not to), and what its inputs and outputs look like. The model reads this docstring to decide whether to call the tool — it never sees the function body.

Mock data is used here so the lab runs without external API access. In production, replace each function body with a live API call.

In [6]:
from langchain_core.tools import tool

# ── Finance tools ────────────────────────────────────────────────────────────

@tool
def get_stock_price(ticker: str) -> str:
    """
    Retrieve the current market price for an equity ticker symbol.
    Use this when you need a live price to answer valuation or portfolio questions.
    Do NOT use this for interest rates or bond yields — use get_rate for those.
    Input: ticker — exchange symbol e.g. AAPL, JPM, GS, PFE, JNJ.
    Returns: current price in USD as a string.
    """
    prices = {"AAPL": 189.45, "MSFT": 415.20, "JPM": 198.73,
              "GS": 452.10, "JNJ": 162.88, "PFE": 29.54, "BAC": 38.12}
    t = ticker.upper().strip()
    p = prices.get(t)
    return f"{t}: ${p:.2f} USD" if p else f"Ticker {t} not found."


@tool
def get_rate(rate_type: str) -> str:
    """
    Look up a benchmark interest rate or bond yield.
    Use this for macroeconomic context, fixed income analysis, or discount rate inputs.
    Supported values for rate_type: fed_funds, sofr, us_10yr, us_2yr, us_30yr.
    Returns: rate as a percentage string.
    """
    rates = {"fed_funds": "5.25%", "sofr": "5.30%",
             "us_10yr": "4.48%", "us_2yr": "4.85%", "us_30yr": "4.62%"}
    key = rate_type.lower().strip()
    r = rates.get(key)
    return f"{rate_type.replace('_',' ').title()}: {r}" if r else (
        f"Rate '{rate_type}' not recognised. Choose from: {', '.join(rates.keys())}.")


@tool
def calculate_portfolio_value(holdings: str) -> str:
    """
    Calculate the total market value of a portfolio from a comma-separated list of TICKER:SHARES pairs.
    Example input: 'AAPL:100,JPM:50,MSFT:200'.
    Use this when asked for a portfolio valuation or total position value.
    Returns: line-by-line breakdown and total value in USD.
    """
    prices = {"AAPL": 189.45, "MSFT": 415.20, "JPM": 198.73,
              "GS": 452.10, "JNJ": 162.88, "PFE": 29.54, "BAC": 38.12}
    total, lines = 0.0, []
    try:
        for pair in holdings.split(","):
            ticker, shares_str = pair.strip().split(":")
            t, s = ticker.strip().upper(), float(shares_str.strip())
            p = prices.get(t, 0.0)
            v = p * s
            total += v
            lines.append(f"  {t}: {s:.0f} x ${p:.2f} = ${v:,.2f}")
    except ValueError as e:
        return f"Parse error: {e}. Format: 'TICKER:SHARES,TICKER:SHARES'"
    return "Portfolio breakdown:\n" + "\n".join(lines) + f"\n\nTotal: ${total:,.2f} USD"


@tool
def get_company_fundamentals(ticker: str) -> str:
    """
    Retrieve key financial fundamentals for a company: P/E ratio, market cap, dividend yield, sector.
    Use this when asked about valuation multiples, sector classification, or income characteristics.
    Input: ticker — e.g. AAPL, JPM, JNJ.
    Returns: structured string with fundamental metrics.
    """
    fundamentals = {
        "AAPL": {"pe": 28.4, "market_cap": "2.95T", "div_yield": "0.55%", "sector": "Technology"},
        "JPM":  {"pe": 11.2, "market_cap": "572B",  "div_yield": "2.30%", "sector": "Financials"},
        "GS":   {"pe": 13.5, "market_cap": "145B",  "div_yield": "2.65%", "sector": "Financials"},
        "JNJ":  {"pe": 15.8, "market_cap": "390B",  "div_yield": "3.10%", "sector": "Healthcare"},
        "PFE":  {"pe": 9.1,  "market_cap": "165B",  "div_yield": "6.20%", "sector": "Healthcare"},
        "BAC":  {"pe": 10.8, "market_cap": "305B",  "div_yield": "2.45%", "sector": "Financials"},
    }
    t = ticker.upper().strip()
    f = fundamentals.get(t)
    if not f:
        return f"Fundamentals not available for {t}."
    return (f"{t} Fundamentals | P/E: {f['pe']} | Market Cap: {f['market_cap']} | "
            f"Dividend Yield: {f['div_yield']} | Sector: {f['sector']}")


# ── Healthcare tools ─────────────────────────────────────────────────────────

@tool
def check_drug_interaction(drug_a: str, drug_b: str) -> str:
    """
    Check for a known clinical drug interaction between two medications.
    Use this when assessing medication safety or when a patient is on multiple prescriptions.
    Input: drug_a, drug_b — generic names of the two medications.
    Returns: interaction severity (MAJOR, MODERATE, MINOR, NONE) and a clinical note.
    """
    db = {
        frozenset(["warfarin",     "aspirin"]):         ("MAJOR",    "Increased bleeding risk. Monitor INR; consider GI prophylaxis."),
        frozenset(["metformin",    "ibuprofen"]):        ("MODERATE", "NSAIDs reduce metformin renal clearance. Risk of lactic acidosis."),
        frozenset(["lisinopril",   "potassium"]):        ("MODERATE", "ACE inhibitors raise serum potassium. Avoid supplements unless hypokalaemia confirmed."),
        frozenset(["atorvastatin", "clarithromycin"]):   ("MAJOR",    "CYP3A4 inhibition raises statin levels. Suspend atorvastatin during antibiotic course."),
        frozenset(["clopidogrel",  "omeprazole"]):       ("MODERATE", "CYP2C19 inhibition reduces clopidogrel activation. Consider pantoprazole instead."),
    }
    key = frozenset([drug_a.lower().strip(), drug_b.lower().strip()])
    result = db.get(key)
    if result:
        sev, note = result
        return f"Severity: {sev} | Note: {note}"
    return f"No major interaction documented between {drug_a} and {drug_b}. Consult a clinical pharmacist for complete assessment."


@tool
def get_clinical_guideline(condition: str) -> str:
    """
    Retrieve the current evidence-based management guideline for a medical condition.
    Use this for treatment protocol questions, first-line therapy selection, or monitoring recommendations.
    Supported conditions: type2_diabetes, hypertension, atrial_fibrillation, heart_failure.
    Returns: structured guideline summary with first-line and escalation options.
    """
    guidelines = {
        "type2_diabetes": (
            "ADA 2024: First-line: Metformin + lifestyle (target HbA1c <7.0%). "
            "Second-line: GLP-1 agonist (ASCVD/weight benefit) or SGLT-2 inhibitor (HF/CKD). "
            "Third-line: Basal insulin if uncontrolled."
        ),
        "hypertension": (
            "ACC/AHA 2017: Target BP <130/80 mmHg. First-line: ACE inhibitor or ARB, "
            "thiazide, or CCB. Avoid ACE inhibitor + ARB combination."
        ),
        "atrial_fibrillation": (
            "ESC 2020: Assess stroke risk (CHA2DS2-VASc). Anticoagulate if score >=2 (male) "
            "or >=3 (female). Prefer NOACs over warfarin unless contraindicated."
        ),
        "heart_failure": (
            "ESC 2021 HFrEF: ACE inhibitor/ARNi + beta-blocker + MRA + SGLT-2 inhibitor. "
            "Target LVEF improvement. ICD if LVEF <=35% on optimal therapy >3 months."
        ),
    }
    key = condition.lower().strip().replace(" ", "_")
    g = guidelines.get(key)
    return g if g else (
        f"No guideline for '{condition}'. Supported: {', '.join(guidelines.keys())}.")


@tool
def get_lab_reference_range(test_name: str) -> str:
    """
    Return the standard adult reference range for a common laboratory test.
    Use this when interpreting lab results or assessing whether a value is within normal limits.
    Supported tests: hba1c, creatinine, potassium, inr, haemoglobin, ldl.
    Returns: reference range with units and clinical note.
    """
    ranges = {
        "hba1c":       "4.0 to 5.6% (normal); 5.7 to 6.4% (pre-diabetes); >=6.5% (diabetes). Target <7.0% in managed T2DM.",
        "creatinine":  "Male: 0.74 to 1.35 mg/dL. Female: 0.59 to 1.04 mg/dL. Elevated values suggest renal impairment.",
        "potassium":   "3.5 to 5.1 mEq/L. Below 3.5: hypokalaemia. Above 5.5: hyperkalaemia (cardiac risk).",
        "inr":         "Therapeutic range for warfarin anticoagulation: 2.0 to 3.0 (general); 2.5 to 3.5 (mechanical heart valves).",
        "haemoglobin": "Male: 13.5 to 17.5 g/dL. Female: 12.0 to 15.5 g/dL. Below lower limit: anaemia classification applies.",
        "ldl":         "Optimal <100 mg/dL. Near optimal 100 to 129. High >=160. Very high >=190. Target <70 in very high CV risk.",
    }
    key = test_name.lower().strip()
    r = ranges.get(key)
    return r if r else f"Reference range not available for '{test_name}'. Supported: {', '.join(ranges.keys())}."


finance_tools    = [get_stock_price, get_rate, calculate_portfolio_value, get_company_fundamentals]
healthcare_tools = [check_drug_interaction, get_clinical_guideline, get_lab_reference_range]

print(f"Finance tools:    {[t.name for t in finance_tools]}")
print(f"Healthcare tools: {[t.name for t in healthcare_tools]}")

Finance tools:    ['get_stock_price', 'get_rate', 'calculate_portfolio_value', 'get_company_fundamentals']
Healthcare tools: ['check_drug_interaction', 'get_clinical_guideline', 'get_lab_reference_range']


## Step 5: Build the Single-Agent Baseline
This is the baseline agent. One model, one tool set, one system prompt, no memory yet. This is the minimal unit from which all further patterns are developed. The system prompt defines persona and compliance posture. It does not instruct the model on which tool to use when — that is the tool descriptions' job.

In [26]:
from langgraph.prebuilt import create_react_agent

FINANCE_SYSTEM_PROMPT = (
    "You are a senior financial analyst at an institutional asset manager. "
    "You have access to real-time market data, rate benchmarks, and fundamental analysis tools. "
    "Always use tools to retrieve current data — never rely on training knowledge for prices or rates. "
    "Present figures with units. Show calculation steps. Close with a one-sentence risk caveat. "
    "This output is informational only and does not constitute investment advice."
)

baseline_agent = create_react_agent(
    model=llm,
    tools=finance_tools,
    prompt=FINANCE_SYSTEM_PROMPT,
)

print(f"Baseline agent nodes: {list(baseline_agent.nodes.keys())}")

# Inspect the Mermaid graph to confirm node/edge structure
print()
print("Graph structure (Mermaid):")
print(baseline_agent.get_graph().draw_mermaid())

Baseline agent nodes: ['__start__', 'agent', 'tools']

Graph structure (Mermaid):
---
config:
  flowchart:
    curve: linear
---
graph TD;
	__start__([<p>__start__</p>]):::first
	agent(agent)
	tools(tools)
	__end__([<p>__end__</p>]):::last
	__start__ --> agent;
	agent -.-> __end__;
	agent -.-> tools;
	tools --> agent;
	classDef default fill:#f2f0ff,line-height:1.2
	classDef first fill-opacity:0
	classDef last fill:#bfb6fc



/tmp/ipykernel_612/1675625727.py:11: LangGraphDeprecatedSinceV10: create_react_agent has been moved to `langchain.agents`. Please update your import to `from langchain.agents import create_agent`. Deprecated in LangGraph V1.0 to be removed in V2.0.
  baseline_agent = create_react_agent(


## Step 6: Sequential Tool Invocation Pattern
In sequential invocation the model calls tools one at a time. Each call informs the next. This query requires the agent to fetch a stock price, then retrieve fundamentals, then fetch a rate — three sequential steps because each contributes context to the final synthesis. Observe in the trace that each tool call waits for its result before the next is issued.

In [8]:
def trace_and_answer(agent, query: str, config: dict = None) -> str:
    """Run an agent, print each step in the reasoning chain, return the final answer."""
    print(f"Query: {query}")
    print("=" * 65)
    invoke_args = {"messages": [HumanMessage(content=query)]}
    result = agent.invoke(invoke_args, config=config) if config else agent.invoke(invoke_args)
    for i, msg in enumerate(result["messages"]):
        mtype = type(msg).__name__
        print(f"  Step {i+1}: {mtype}")
        if hasattr(msg, "tool_calls") and msg.tool_calls:
            for tc in msg.tool_calls:
                print(f"    Tool: {tc['name']}  Args: {tc['args']}")
        elif hasattr(msg, "content") and msg.content:
            preview = str(msg.content)[:180]
            print(f"    Content: {preview}{'...' if len(str(msg.content)) > 180 else ''}")
    print()
    final = result["messages"][-1].content
    print("ANSWER:")
    print(final)
    return final


# Sequential pattern: price -> fundamentals -> rate -> synthesis
trace_and_answer(
    baseline_agent,
    "Give me the current price and key fundamentals for JPM, then tell me the current Fed Funds rate "
    "and explain what rising rates mean for JPM's net interest margin."
)

Query: Give me the current price and key fundamentals for JPM, then tell me the current Fed Funds rate and explain what rising rates mean for JPM's net interest margin.
  Step 1: HumanMessage
    Content: Give me the current price and key fundamentals for JPM, then tell me the current Fed Funds rate and explain what rising rates mean for JPM's net interest margin.
  Step 2: AIMessage
    Tool: get_stock_price  Args: {'ticker': 'JPM'}
    Tool: get_company_fundamentals  Args: {'ticker': 'JPM'}
    Tool: get_rate  Args: {'rate_type': 'fed_funds'}
  Step 3: ToolMessage
    Content: JPM: $198.73 USD
  Step 4: ToolMessage
    Content: JPM Fundamentals | P/E: 11.2 | Market Cap: 572B | Dividend Yield: 2.30% | Sector: Financials
  Step 5: ToolMessage
    Content: Fed Funds: 5.25%
  Step 6: AIMessage
    Content: ### Current Price and Fundamentals for JPMorgan Chase (JPM):
- **Price**: $198.73 USD
- **P/E Ratio**: 11.2
- **Market Cap**: $572 billion
- **Dividend Yield**: 2.30%
- **Sector**:...


"### Current Price and Fundamentals for JPMorgan Chase (JPM):\n- **Price**: $198.73 USD\n- **P/E Ratio**: 11.2\n- **Market Cap**: $572 billion\n- **Dividend Yield**: 2.30%\n- **Sector**: Financials\n\n### Current Fed Funds Rate:\n- **Rate**: 5.25%\n\n### Impact of Rising Rates on JPM's Net Interest Margin:\nRising interest rates, such as the Fed Funds rate, generally benefit banks like JPMorgan Chase by increasing their net interest margin (NIM). NIM represents the difference between the interest income generated from loans and the interest paid on deposits. As rates rise, banks can charge higher interest on loans while deposit rates typically lag, widening the margin. However, this benefit may be offset if higher rates lead to reduced loan demand or increased defaults.\n\n**Risk Caveat**: Interest rate impacts depend on broader economic conditions, loan portfolio composition, and deposit sensitivity."

## Step 7: Parallel Tool Invocation Pattern
GPT-4o returns multiple tool calls in a single response when it determines they are independent. LangGraph's `ToolNode` executes them concurrently. This query asks for prices of three independent tickers. Observe in the trace that a single `AIMessage` step contains multiple `tool_calls` entries, and a single `ToolMessage` step returns all three results before the model continues.

In [9]:
# Parallel pattern: three independent price lookups in a single model turn
trace_and_answer(
    baseline_agent,
    "I need the current prices for AAPL, JPM, and GS simultaneously. "
    "Then calculate the total value of holding 100 shares of each."
)

Query: I need the current prices for AAPL, JPM, and GS simultaneously. Then calculate the total value of holding 100 shares of each.
  Step 1: HumanMessage
    Content: I need the current prices for AAPL, JPM, and GS simultaneously. Then calculate the total value of holding 100 shares of each.
  Step 2: AIMessage
    Tool: get_stock_price  Args: {'ticker': 'AAPL'}
    Tool: get_stock_price  Args: {'ticker': 'JPM'}
    Tool: get_stock_price  Args: {'ticker': 'GS'}
  Step 3: ToolMessage
    Content: AAPL: $189.45 USD
  Step 4: ToolMessage
    Content: JPM: $198.73 USD
  Step 5: ToolMessage
    Content: GS: $452.10 USD
  Step 6: AIMessage
    Tool: calculate_portfolio_value  Args: {'holdings': 'AAPL:100,JPM:100,GS:100'}
  Step 7: ToolMessage
    Content: Portfolio breakdown:
  AAPL: 100 x $189.45 = $18,945.00
  JPM: 100 x $198.73 = $19,873.00
  GS: 100 x $452.10 = $45,210.00

Total: $84,028.00 USD
  Step 8: AIMessage
    Content: The current stock prices are:
- AAPL: $189.45 USD
- JPM: $1

'The current stock prices are:\n- AAPL: $189.45 USD\n- JPM: $198.73 USD\n- GS: $452.10 USD\n\nFor a portfolio holding 100 shares of each:\n- AAPL: 100 x $189.45 = $18,945.00\n- JPM: 100 x $198.73 = $19,873.00\n- GS: 100 x $452.10 = $45,210.00\n\n**Total Portfolio Value:** $84,028.00 USD.\n\nRisk caveat: Market prices are subject to change and may impact portfolio valuation.'

## Step 8: Conditional Tool Invocation Pattern
A well-calibrated agent calls tools only when external data is genuinely needed. This query can be answered from the rates data alone — it does not require a stock price lookup. The agent should invoke `get_rate` once and answer directly. If the agent is calling tools unnecessarily, the system prompt or tool descriptions need tightening.

In [10]:
# Conditional pattern: only one tool is needed; unnecessary calls indicate a prompt quality issue
trace_and_answer(
    baseline_agent,
    "What is the current yield spread between the US 10-year Treasury and the 2-year Treasury, "
    "and what does an inverted yield curve signal about the economic cycle?"
)

Query: What is the current yield spread between the US 10-year Treasury and the 2-year Treasury, and what does an inverted yield curve signal about the economic cycle?
  Step 1: HumanMessage
    Content: What is the current yield spread between the US 10-year Treasury and the 2-year Treasury, and what does an inverted yield curve signal about the economic cycle?
  Step 2: AIMessage
    Tool: get_rate  Args: {'rate_type': 'us_10yr'}
    Tool: get_rate  Args: {'rate_type': 'us_2yr'}
  Step 3: ToolMessage
    Content: Us 10Yr: 4.48%
  Step 4: ToolMessage
    Content: Us 2Yr: 4.85%
  Step 5: AIMessage
    Content: The current yield on the US 10-year Treasury is 4.48%, and the yield on the US 2-year Treasury is 4.85%. The yield spread is calculated as:

\[
\text{Yield Spread} = \text{10-year ...

ANSWER:
The current yield on the US 10-year Treasury is 4.48%, and the yield on the US 2-year Treasury is 4.85%. The yield spread is calculated as:

\[
\text{Yield Spread} = \text{10-year yield} - 

'The current yield on the US 10-year Treasury is 4.48%, and the yield on the US 2-year Treasury is 4.85%. The yield spread is calculated as:\n\n\\[\n\\text{Yield Spread} = \\text{10-year yield} - \\text{2-year yield}\n\\]\n\n\\[\n\\text{Yield Spread} = 4.48\\% - 4.85\\% = -0.37\\%\n\\]\n\nThis indicates an inverted yield curve, where short-term rates exceed long-term rates.\n\nAn inverted yield curve often signals market expectations of an economic slowdown or recession. It reflects investor concerns about future growth, leading to higher demand for long-term bonds and lower yields on those instruments.\n\n**Risk Caveat:** Yield curve analysis is a macroeconomic indicator and should not be used in isolation for investment decisions.'

In [11]:
# No-tool path: pure reasoning, no external data required
# A good agent should answer this directly without any tool call
trace_and_answer(
    baseline_agent,
    "What is 12% of a $4,500,000 portfolio?"
)

Query: What is 12% of a $4,500,000 portfolio?
  Step 1: HumanMessage
    Content: What is 12% of a $4,500,000 portfolio?
  Step 2: AIMessage
    Content: To calculate 12% of a $4,500,000 portfolio:

\[
12\% \times 4,500,000 = 0.12 \times 4,500,000 = 540,000
\]

**Result:** 12% of a $4,500,000 portfolio is **$540,000**.

**Risk Cavea...

ANSWER:
To calculate 12% of a $4,500,000 portfolio:

\[
12\% \times 4,500,000 = 0.12 \times 4,500,000 = 540,000
\]

**Result:** 12% of a $4,500,000 portfolio is **$540,000**.

**Risk Caveat:** Ensure portfolio allocations align with your risk tolerance and investment objectives.


'To calculate 12% of a $4,500,000 portfolio:\n\n\\[\n12\\% \\times 4,500,000 = 0.12 \\times 4,500,000 = 540,000\n\\]\n\n**Result:** 12% of a $4,500,000 portfolio is **$540,000**.\n\n**Risk Caveat:** Ensure portfolio allocations align with your risk tolerance and investment objectives.'

## Step 9: Threading — Adding Stateful Memory to the Baseline
The baseline agent above is stateless. Each call is independent. Here the same agent is given a `MemorySaver` checkpointer and a `thread_id`. All three turns below are part of one conversation. The agent resolves references like "that portfolio" and "those two stocks" from the thread history without the user restating them.

This is the Assistants pattern at its core: an identity (system prompt), capabilities (tools), and a thread (conversation history) working together.

In [12]:
from langgraph.checkpoint.memory import MemorySaver
import uuid

stateful_agent = create_react_agent(
    model=llm,
    tools=finance_tools,
    prompt=FINANCE_SYSTEM_PROMPT,
    checkpointer=MemorySaver(),
)

# Single session thread — all turns share this config
session = {"configurable": {"thread_id": str(uuid.uuid4())}}

def turn(agent, message: str, config: dict) -> str:
    result = agent.invoke(
        {"messages": [HumanMessage(content=message)]}, config=config
    )
    return result["messages"][-1].content


print("TURN 1")
q1 = "Get me the current price and fundamentals for BAC and JPM."
print(f"User:  {q1}")
r1 = turn(stateful_agent, q1, session)
print(f"Agent: {r1}\n")

TURN 1
User:  Get me the current price and fundamentals for BAC and JPM.


/tmp/ipykernel_612/1446491586.py:4: LangGraphDeprecatedSinceV10: create_react_agent has been moved to `langchain.agents`. Please update your import to `from langchain.agents import create_agent`. Deprecated in LangGraph V1.0 to be removed in V2.0.
  stateful_agent = create_react_agent(


Agent: Here are the current prices and fundamentals for BAC and JPM:

- **Bank of America (BAC):**
  - Current Price: $38.12 USD
  - Fundamentals: P/E Ratio: 10.8, Market Cap: $305 billion, Dividend Yield: 2.45%, Sector: Financials

- **JPMorgan Chase (JPM):**
  - Current Price: $198.73 USD
  - Fundamentals: P/E Ratio: 11.2, Market Cap: $572 billion, Dividend Yield: 2.30%, Sector: Financials

**Risk Caveat:** Market conditions can change rapidly; verify data before making investment decisions.



In [13]:
print("TURN 2 — references 'those two banks' from thread history")
q2 = "If I hold 500 shares of each of those two banks, what is my total position value?"
print(f"User:  {q2}")
r2 = turn(stateful_agent, q2, session)
print(f"Agent: {r2}\n")

TURN 2 — references 'those two banks' from thread history
User:  If I hold 500 shares of each of those two banks, what is my total position value?
Agent: Your total position value for holding 500 shares of each bank is:

- **Bank of America (BAC):** 500 shares x $38.12 = $19,060.00 USD
- **JPMorgan Chase (JPM):** 500 shares x $198.73 = $99,365.00 USD

**Total Portfolio Value:** $118,425.00 USD

**Risk Caveat:** Equity prices fluctuate; confirm values before making portfolio decisions.



In [14]:
print("TURN 3 — contextualises against prior portfolio and adds rate data")
q3 = (
    "Given the current Fed Funds rate and the sector classification you retrieved earlier, "
    "how exposed is that portfolio to interest rate risk?"
)
print(f"User:  {q3}")
r3 = turn(stateful_agent, q3, session)
print(f"Agent: {r3}")

TURN 3 — contextualises against prior portfolio and adds rate data
User:  Given the current Fed Funds rate and the sector classification you retrieved earlier, how exposed is that portfolio to interest rate risk?
Agent: The current Fed Funds rate is 5.25%, and both Bank of America (BAC) and JPMorgan Chase (JPM) are classified under the Financials sector. Financial institutions, particularly banks, are highly sensitive to interest rate changes due to their reliance on net interest margins (NIM) — the difference between the interest earned on loans and the interest paid on deposits.

**Interest Rate Risk Exposure:**
- **Positive Impact:** Rising rates can increase NIM, benefiting banks' profitability.
- **Negative Impact:** Higher rates may reduce loan demand and increase default risk, potentially impacting earnings.

Given the portfolio's exclusive focus on banks, it is significantly exposed to interest rate risk, both positively and negatively, depending on the broader economic environ

## Step 10: Thread Isolation — Confirming Session Boundaries
A new `thread_id` produces a completely independent conversation. The agent has no access to the BAC/JPM discussion above. This is the isolation guarantee that makes multi-tenant applications safe — one user's thread cannot read another's.

In [15]:
new_session = {"configurable": {"thread_id": str(uuid.uuid4())}}

print("NEW THREAD — no access to the previous conversation")
q = "What were the two bank stocks I asked about earlier?"
print(f"User:  {q}")
r = turn(stateful_agent, q, new_session)
print(f"Agent: {r}")

NEW THREAD — no access to the previous conversation
User:  What were the two bank stocks I asked about earlier?
Agent: I don't have memory of prior interactions, so I can't recall the two bank stocks you mentioned earlier. If you provide the ticker symbols or names of the banks, I can assist you further.


## Step 11: Healthcare Baseline Agent
A second baseline agent with a healthcare tool set and system prompt. This agent is architecturally identical to the finance agent — same model, same LangGraph runtime, same invocation patterns. Only the tools and system prompt differ. This is exactly how specialists are composed in a multi-agent supervisor architecture: each is a baseline agent with a focused scope.

In [16]:
HEALTHCARE_SYSTEM_PROMPT = (
    "You are a clinical decision-support assistant for qualified healthcare professionals. "
    "You have access to drug interaction databases, clinical guidelines, and lab reference ranges. "
    "Always retrieve data from tools rather than training knowledge — guidelines are updated regularly. "
    "Structure responses clearly. Always state that clinical decisions require individual patient assessment "
    "by a qualified clinician. This tool provides informational support only."
)

healthcare_agent = create_react_agent(
    model=llm,
    tools=healthcare_tools,
    prompt=HEALTHCARE_SYSTEM_PROMPT,
)

print("Healthcare baseline agent created.")

Healthcare baseline agent created.


/tmp/ipykernel_612/390553969.py:9: LangGraphDeprecatedSinceV10: create_react_agent has been moved to `langchain.agents`. Please update your import to `from langchain.agents import create_agent`. Deprecated in LangGraph V1.0 to be removed in V2.0.
  healthcare_agent = create_react_agent(


In [17]:
# Sequential: guideline lookup -> interaction check -> lab reference
trace_and_answer(
    healthcare_agent,
    "A patient with type 2 diabetes and heart failure is on metformin and warfarin. "
    "What do the guidelines say about managing each condition, is there an interaction between their medications, "
    "and what is the therapeutic INR range for someone on warfarin?"
)

Query: A patient with type 2 diabetes and heart failure is on metformin and warfarin. What do the guidelines say about managing each condition, is there an interaction between their medications, and what is the therapeutic INR range for someone on warfarin?
  Step 1: HumanMessage
    Content: A patient with type 2 diabetes and heart failure is on metformin and warfarin. What do the guidelines say about managing each condition, is there an interaction between their medic...
  Step 2: AIMessage
    Tool: get_clinical_guideline  Args: {'condition': 'type2_diabetes'}
    Tool: get_clinical_guideline  Args: {'condition': 'heart_failure'}
    Tool: check_drug_interaction  Args: {'drug_a': 'metformin', 'drug_b': 'warfarin'}
    Tool: get_lab_reference_range  Args: {'test_name': 'inr'}
  Step 3: ToolMessage
    Content: ADA 2024: First-line: Metformin + lifestyle (target HbA1c <7.0%). Second-line: GLP-1 agonist (ASCVD/weight benefit) or SGLT-2 inhibitor (HF/CKD). Third-line: Basal insulin if u

'Here is the information based on your query:\n\n### 1. **Management Guidelines**\n   - **Type 2 Diabetes (ADA 2024 Guidelines):**\n     - **First-line therapy:** Metformin combined with lifestyle modifications, aiming for an HbA1c target of <7.0%.\n     - **Second-line therapy:** \n       - GLP-1 receptor agonists for cardiovascular or weight benefits.\n       - SGLT-2 inhibitors for patients with heart failure (HF) or chronic kidney disease (CKD).\n     - **Third-line therapy:** Basal insulin if glycemic control remains inadequate.\n\n   - **Heart Failure (ESC 2021 Guidelines for HFrEF):**\n     - Core therapy includes:\n       - ACE inhibitors or ARNi (angiotensin receptor-neprilysin inhibitors).\n       - Beta-blockers.\n       - MRAs (mineralocorticoid receptor antagonists).\n       - SGLT-2 inhibitors (recommended for heart failure management).\n     - Goal: Improve left ventricular ejection fraction (LVEF).\n     - Consider ICD (implantable cardioverter-defibrillator) if LVEF re

## Step 12: Streaming — Token-Level and Step-Level Output
Two streaming modes are demonstrated here.

`stream_mode="messages"` delivers tokens as they arrive from the model, enabling real-time display in a UI. This is the mode used in consumer-facing chat interfaces and in GitHub Copilot Agent Mode's streaming response panel.

`stream_mode="values"` delivers the full state after each graph node completes. This is the mode used for audit logging, compliance recording, and step-by-step debugging — the equivalent of a full reasoning transcript.

In [18]:
# Token-level streaming
stream_query = (
    "Fetch the fundamentals for PFE and JNJ, then compare their dividend yields "
    "against the current 10-year Treasury yield and give a view on relative income attractiveness."
)

stream_cfg = {"configurable": {"thread_id": str(uuid.uuid4())}}

print("Token streaming (stream_mode='messages'):")
print("-" * 65)

for chunk, _ in stateful_agent.stream(
    {"messages": [HumanMessage(content=stream_query)]},
    config=stream_cfg,
    stream_mode="messages"
):
    if (
        hasattr(chunk, "content") and chunk.content
        and not getattr(chunk, "tool_calls", None)
        and not getattr(chunk, "tool_call_chunks", None)
    ):
        print(chunk.content, end="", flush=True)

print("\n" + "-" * 65)

Token streaming (stream_mode='messages'):
-----------------------------------------------------------------
PFE Fundamentals | P/E: 9.1 | Market Cap: 165B | Dividend Yield: 6.20% | Sector: HealthcareJNJ Fundamentals | P/E: 15.8 | Market Cap: 390B | Dividend Yield: 3.10% | Sector: HealthcareUs 10Yr: 4.48%Here are the fundamentals and dividend yields for Pfizer (PFE) and Johnson & Johnson (JNJ), compared against the current 10-year Treasury yield:

1. **Pfizer (PFE)**:
   - P/E Ratio: 9.1
   - Market Cap: $165 billion
   - Dividend Yield: **6.20%**
   - Sector: Healthcare

2. **Johnson & Johnson (JNJ)**:
   - P/E Ratio: 15.8
   - Market Cap: $390 billion
   - Dividend Yield: **3.10%**
   - Sector: Healthcare

3. **10-Year Treasury Yield**: **4.48%**

### Relative Income Attractiveness:
- **Pfizer (PFE)** offers a significantly higher dividend yield (6.20%) compared to the 10-year Treasury yield (4.48%), making it more attractive for income-focused investors seeking higher returns, albeit

In [19]:
# Step-level streaming — audit trace
audit_query = (
    "What is the current SOFR rate and what does it mean for floating-rate loan portfolios at commercial banks?"
)

audit_cfg = {"configurable": {"thread_id": str(uuid.uuid4())}}

print("Step-level audit trace (stream_mode='values'):")
print("=" * 65)

for step, state in enumerate(
    stateful_agent.stream(
        {"messages": [HumanMessage(content=audit_query)]},
        config=audit_cfg,
        stream_mode="values"
    )
):
    latest = state["messages"][-1]
    print(f"Step {step+1}: {type(latest).__name__}")
    if hasattr(latest, "tool_calls") and latest.tool_calls:
        for tc in latest.tool_calls:
            print(f"  Tool: {tc['name']}  Args: {tc['args']}")
    elif hasattr(latest, "content") and latest.content:
        print(f"  Content: {str(latest.content)[:200]}")
    print()

Step-level audit trace (stream_mode='values'):
Step 1: HumanMessage
  Content: What is the current SOFR rate and what does it mean for floating-rate loan portfolios at commercial banks?

Step 2: AIMessage
  Tool: get_rate  Args: {'rate_type': 'sofr'}

Step 3: ToolMessage
  Content: Sofr: 5.30%

Step 4: AIMessage
  Content: The current Secured Overnight Financing Rate (SOFR) is 5.30%. 

SOFR is a benchmark interest rate used for pricing floating-rate loans and other financial instruments. For commercial banks, a higher S



## Step 13: Token Usage Tracking
Every model call in an agent loop consumes tokens. Because the full conversation history is resent with each call, token consumption grows with thread length. A 10-turn conversation can easily consume 5x the tokens of the same information delivered in a single call.

Token tracking serves three purposes in production: cost allocation (which team or workflow is consuming the most), performance tuning (identify turns where the context has grown unnecessarily large), and rate limit management (Azure OpenAI enforces tokens-per-minute limits per deployment).

In [20]:
token_cfg = {"configurable": {"thread_id": str(uuid.uuid4())}}

result = stateful_agent.invoke(
    {"messages": [HumanMessage(
        content="Get the price and fundamentals for GS, then fetch the US 10-year yield "
                "and comment on whether GS's P/E is attractive relative to the risk-free rate."
    )]},
    config=token_cfg
)

print("Token usage breakdown:")
print("-" * 55)

total_in, total_out = 0, 0
for msg in result["messages"]:
    usage = getattr(msg, "response_metadata", {}).get("token_usage", {})
    if usage:
        inp = usage.get("prompt_tokens", 0)
        out = usage.get("completion_tokens", 0)
        total_in  += inp
        total_out += out
        print(f"  {type(msg).__name__:<25} in={inp:>5}  out={out:>5}")

print("-" * 55)
print(f"  {'Total':<25} in={total_in:>5}  out={total_out:>5}")
print(f"  Grand total: {total_in + total_out} tokens")
print()
print("Final answer:")
print(result["messages"][-1].content)

Token usage breakdown:
-------------------------------------------------------
  AIMessage                 in=  458  out=   68
  AIMessage                 in=  589  out=  205
-------------------------------------------------------
  Total                     in= 1047  out=  273
  Grand total: 1320 tokens

Final answer:
The current price of Goldman Sachs (GS) is $452.10 USD. Its fundamentals are as follows: P/E ratio of 13.5, market capitalization of $145 billion, dividend yield of 2.65%, and it operates in the Financials sector. The current yield on the US 10-year Treasury is 4.48%.

### Analysis:
The P/E ratio of 13.5 implies an earnings yield of approximately **7.41%** (calculated as 1 / P/E). Comparing this to the risk-free rate of 4.48% (US 10-year yield), GS's earnings yield offers a premium of **2.93%** over the risk-free rate. This suggests that GS may be attractively valued relative to the risk-free rate, assuming its earnings are stable and growth prospects are favorable.

###

## Step 14: Recursion Limit and Error Resilience
The recursion limit is the circuit breaker for runaway agent loops. If the model enters a state where it keeps calling tools without converging on a final answer — a failure mode that can happen when tool descriptions are ambiguous or the user's query is impossible to satisfy — the recursion limit stops execution after a fixed number of steps.

The default is 25 steps. For a focused single-agent baseline with 3 to 7 tools, a limit of 10 is usually sufficient and prevents excessive token consumption from looping agents.

In [21]:
# Test 1: Unknown ticker — agent should surface the tool error gracefully
print("Test: unknown ticker")
print("-" * 55)
rc = {"configurable": {"thread_id": str(uuid.uuid4())}, "recursion_limit": 10}
r = stateful_agent.invoke(
    {"messages": [HumanMessage(
        content="What is the price and fundamentals for XYZINVALID?"
    )]},
    config=rc
)
print(r["messages"][-1].content)
print()

Test: unknown ticker
-------------------------------------------------------
The ticker symbol "XYZINVALID" is not recognized in the market data system, and no fundamentals are available for it. Please verify the ticker symbol and try again.

Risk caveat: Ensure the accuracy of ticker symbols to avoid misinterpretation or errors in financial analysis.



In [22]:
# Test 2: Pure arithmetic — agent should answer without calling any tool
print("Test: pure arithmetic — no tool call expected")
print("-" * 55)
rc2 = {"configurable": {"thread_id": str(uuid.uuid4())}, "recursion_limit": 10}
r2 = stateful_agent.invoke(
    {"messages": [HumanMessage(
        content="A bond has a face value of $1,000,000 and a coupon rate of 6.5%. What is the annual coupon payment?"
    )]},
    config=rc2
)
steps = len(r2["messages"])
tool_calls_made = sum(
    1 for m in r2["messages"]
    if hasattr(m, "tool_calls") and m.tool_calls
)
print(f"Total steps: {steps} | Tool calls made: {tool_calls_made}")
print(r2["messages"][-1].content)

Test: pure arithmetic — no tool call expected
-------------------------------------------------------
Total steps: 2 | Tool calls made: 0
The annual coupon payment for a bond is calculated using the formula:

\[
\text{Annual Coupon Payment} = \text{Face Value} \times \text{Coupon Rate}
\]

Given:
- Face Value = $1,000,000
- Coupon Rate = 6.5% = 0.065 (as a decimal)

\[
\text{Annual Coupon Payment} = 1,000,000 \times 0.065 = 65,000 \, \text{USD}
\]

### Final Answer:
The annual coupon payment is **$65,000 USD**.

### Risk Caveat:
Bond payments are subject to credit risk, and the issuer's ability to meet coupon obligations may vary.


## Step 15: Comparing Stateless vs Stateful Behaviour
This cell runs the same follow-up query against both the stateless `baseline_agent` and the stateful `stateful_agent`. The stateless agent cannot resolve "those two stocks" and must ask for clarification or refuse. The stateful agent resolves it from the thread. This comparison makes the value of threading concrete and measurable.

In [23]:
# Prime the stateful agent with context
prime_session = {"configurable": {"thread_id": str(uuid.uuid4())}}
turn(stateful_agent, "What are the current prices for AAPL and MSFT?", prime_session)

followup = "What would it cost to buy 250 shares of each of those two stocks?"

print("STATELESS AGENT (no memory of prior turn):")
print("-" * 55)
stateless_result = baseline_agent.invoke(
    {"messages": [HumanMessage(content=followup)]}
)
print(stateless_result["messages"][-1].content)
print()

print("STATEFUL AGENT (resolves from thread history):")
print("-" * 55)
stateful_result = turn(stateful_agent, followup, prime_session)
print(stateful_result)

STATELESS AGENT (no memory of prior turn):
-------------------------------------------------------
To calculate the cost of buying 250 shares of each of the two stocks, I need the ticker symbols for the stocks you're referring to. Could you please provide them?

STATEFUL AGENT (resolves from thread history):
-------------------------------------------------------
To calculate the cost of buying 250 shares of each stock:

1. **AAPL**: Current price = $189.45 USD  
   Cost for 250 shares = \( 250 \times 189.45 = 47,362.50 \, \text{USD} \)

2. **MSFT**: Current price = $415.20 USD  
   Cost for 250 shares = \( 250 \times 415.20 = 103,800.00 \, \text{USD} \)

**Total cost** = \( 47,362.50 + 103,800.00 = 151,162.50 \, \text{USD} \)

It would cost $151,162.50 USD to buy 250 shares of both AAPL and MSFT.

Risk caveat: Stock prices can fluctuate rapidly, and transaction costs such as brokerage fees are not included in this calculation.


## Step 16: Preparing for Multi-Agent Composition
This step demonstrates what a supervisor agent needs from a specialist agent. The finance and healthcare baseline agents are each wrapped in a simple callable that a supervisor can route to by name. This pattern — each specialist as a callable with a clear input/output contract — is the bridge from single-agent baseline to multi-agent architecture.

Neither agent has memory here because in a multi-agent system the supervisor typically owns the thread state and passes it to specialists as needed.

In [24]:
def run_finance_specialist(query: str) -> str:
    """Route a query to the finance baseline agent and return the final answer."""
    result = baseline_agent.invoke({"messages": [HumanMessage(content=query)]})
    return result["messages"][-1].content


def run_healthcare_specialist(query: str) -> str:
    """Route a query to the healthcare baseline agent and return the final answer."""
    result = healthcare_agent.invoke({"messages": [HumanMessage(content=query)]})
    return result["messages"][-1].content


SPECIALIST_REGISTRY = {
    "finance":    run_finance_specialist,
    "healthcare": run_healthcare_specialist,
}

print("Specialist registry:")
for name, fn in SPECIALIST_REGISTRY.items():
    print(f"  {name}: {fn.__doc__.strip()}")

print()

# Simulate supervisor routing: finance query -> finance specialist
finance_query = "What are the current prices and dividend yields for JNJ and PFE?"
print(f"Routing to finance specialist: '{finance_query}'")
print("-" * 65)
print(SPECIALIST_REGISTRY["finance"](finance_query))
print()

# Simulate supervisor routing: healthcare query -> healthcare specialist
health_query = "Is there a known interaction between clopidogrel and omeprazole?"
print(f"Routing to healthcare specialist: '{health_query}'")
print("-" * 65)
print(SPECIALIST_REGISTRY["healthcare"](health_query))

Specialist registry:
  finance: Route a query to the finance baseline agent and return the final answer.
  healthcare: Route a query to the healthcare baseline agent and return the final answer.

Routing to finance specialist: 'What are the current prices and dividend yields for JNJ and PFE?'
-----------------------------------------------------------------
The current stock price for Johnson & Johnson (JNJ) is $162.88 USD, and its dividend yield is 3.10%. For Pfizer (PFE), the current stock price is $29.54 USD, and its dividend yield is 6.20%.

Risk caveat: Dividend yields and stock prices are subject to market fluctuations and may change rapidly.

Routing to healthcare specialist: 'Is there a known interaction between clopidogrel and omeprazole?'
-----------------------------------------------------------------
There is a **moderate interaction** between clopidogrel and omeprazole. Omeprazole inhibits the enzyme CYP2C19, which is necessary for the activation of clopidogrel. This can 

## Summary: What was built and what it means
This lab constructed and validated a single-agent baseline across five theoretical dimensions and sixteen implementation steps.

**Assistants and Tool Calling** — An Assistant is a model with identity (system prompt), capabilities (tools), and history (thread). Tool calling is a request protocol, not an execution protocol. The model produces a structured JSON object; your code decides whether to honour it. This separation is the foundation of human-in-the-loop control.

**Threading** — A thread is the unit of conversation state, identified by a `thread_id` and persisted by a checkpointer. Threads are how Assistants maintain context across turns. Thread isolation prevents state leaking between sessions. In production, `MemorySaver` is replaced with a database-backed checkpointer to survive process restarts and satisfy audit requirements.

**Tool Invocation Patterns** — Sequential invocation is the default, safe, and auditable pattern. Parallel invocation is available when the model identifies independent sub-tasks, reducing latency without additional code. Conditional invocation is an emergent behaviour — a well-designed agent calls tools only when necessary. The recursion limit is the circuit breaker for all patterns.

**Multi-Agent Architectures** — Every agent in a multi-agent system is a single-agent baseline with a scoped system prompt and a curated tool set. The baseline must be reliable before it can be composed. The specialist registry pattern in Step 16 is the direct precursor to a supervisor agent.

**MCP and GitHub Copilot Agent Mode** — MCP standardises the tool interface so that any MCP-compliant agent can use any MCP server. The `@tool` schema produced by LangChain is structurally equivalent to an MCP tool definition. GitHub Copilot Agent Mode is architecturally identical to the baseline built here: system prompt, thread, tools, and a ReAct loop. Debugging a stuck Copilot agent session uses the same techniques as debugging the agents in this lab: inspect the thread, check tool descriptions, add a recursion limit.